In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from helper import DataLoader
import models.SimpleGCN.SimpleGCN as SimpleGCN
import torch
from sklearn.metrics import root_mean_squared_error

In [3]:
DATAPATH = "../data"
RATING_DATA = DATAPATH + "/train_ratings.csv"
TBR_DATA = DATAPATH + "/train_tbr.csv"

In [4]:
data_manager = DataLoader.DataLoader(data_dir=DATAPATH)
train_df, valid_df = data_manager.load_and_split()

In [5]:
print(train_df)

         sid  pid  rating
0          0    1       5
1          0   13       4
2          0   22       5
3          0   26       4
4          0   62       4
...      ...  ...     ...
896738   957  337       3
896739  2218  232       5
896740  1146  255       4
896741  5758  519       4
896742  1062  437       5

[896743 rows x 3 columns]


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
edge_t0_index, edge_t0_weights, edge_t1_index, edge_t1_weights = data_manager.get_graph_tensors(device)

In [8]:
rating_model = SimpleGCN.SimpleGCN(data_manager.num_users, data_manager.num_items, embedding_dim=128, dropout=0.2, num_layers=1, layer_type="LightGCN", use_mlp_head=True, mlp_hidden_dims=[256, 64])
rating_model = rating_model.to(device)
def rating_val():
    rating_model.eval()
    with torch.no_grad():
        # 1. Pass the IDs AND the graph tensors
        val_preds = rating_model.predict_ratings(
            valid_df["sid"].values, 
            valid_df["pid"].values,
            edge_t0_index,    # The graph structure
            edge_t0_weights   # The graph weights
        )
        
        # 2. Ensure variable names match (val_preds)
        return root_mean_squared_error(valid_df["rating"].values, val_preds * 5)




In [9]:
rating_model.fit(
    edge_index=edge_t0_index,
    weights=edge_t0_weights,
    log_every=50,
    lr=1e-3,
    val_fn=rating_val,
    targets=edge_t0_weights.float(),
    lambda_reg=1e-10,
    early_stop_patience=200,
    scheduler_patience=25
)

Epoch     0 | Loss: 0.1113 | Task: 0.1113 | Monitor: 1.6506 | LR: 1.00e-03 | No improvement: 0/200


KeyboardInterrupt: 